# Build & Run GPU Docker Container on Colab

This notebook builds or pulls the LightningPoseTrack GPU Docker image and runs Jupyter inside the container with GPU passthrough.

**Runtime**: Python 3, **GPU** accelerator (T4, V100, or A100)

**Workflow**:
1. Clone repo and install Docker
2. Pull pre-built GPU image from Docker Hub (30s) or build from source (15min)
3. Launch Jupyter inside the container
4. Connect VS Code to the remote container (optional)

In [ ]:
# ===== CONFIGURATION =====
DOCKER_HUB_USER = "kaarthikbalakrishnan"
IMAGE_NAME = f"{DOCKER_HUB_USER}/lightningposetrack"
IMAGE_TAG = "latest"
FULL_IMAGE = f"{IMAGE_NAME}:{IMAGE_TAG}"

# Source: build from repo (True) or pull from Docker Hub (False)
BUILD_LOCALLY = False  # Set True to build from source (slow)

# Port for Jupyter inside container
JUPYTER_PORT = 8888

# Repo URL
GITHUB_REPO = "https://github.com/kaarthik-balakrishnan/LightningPoseTrack.git"
GIT_BRANCH = "main"

## Step 1: Mount Google Drive & Clone Repo

In [ ]:
import os
from pathlib import Path

# Mount Drive for persistent data
from google.colab import drive
drive.mount("/content/drive")

# Clone / pull repo
REPO_DIR = "/content/LightningPoseTrack"
if not os.path.exists(REPO_DIR):
    !git clone -b {GIT_BRANCH} {GITHUB_REPO} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}

print(f"Repo ready at {REPO_DIR}")

## Step 2: Install Docker on Colab VM

Colab VMs do not have Docker by default. This cell installs it.

In [ ]:
import subprocess
import sys

def check_docker():
    try:
        subprocess.run(["docker", "--version"], capture_output=True, check=True)
        return True
    except (FileNotFoundError, subprocess.CalledProcessError):
        return False

if not check_docker():
    print("Installing Docker...")
    !apt-get update -qq && apt-get install -y -qq ca-certificates curl > /dev/null 2>&1
    !curl -fsSL https://get.docker.com -o /tmp/get-docker.sh
    !sh /tmp/get-docker.sh > /dev/null 2>&1
    print("Docker installed.")
else:
    print(f"Docker already available.")

!docker --version

# Verify NVIDIA container toolkit
try:
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    if result.returncode == 0:
        print("NVIDIA drivers detected:")
        for line in result.stdout.split('\n')[:3]:
            print(f"  {line}")
    else:
        print("WARNING: nvidia-smi failed. GPU may not passthrough.")
except FileNotFoundError:
    print("WARNING: nvidia-smi not found. GPU may not passthrough.")

## Step 3: Pull or Build the GPU Image

In [ ]:
import time

if BUILD_LOCALLY:
    print("Building GPU Docker image from source (this takes 15-20 minutes)...")
    start = time.time()
    !docker build -t {FULL_IMAGE} -f .devcontainer/Dockerfile.gpu .
    elapsed = time.time() - start
    print(f"Build complete in {elapsed/60:.1f} minutes.")
else:
    print(f"Pulling {FULL_IMAGE} from Docker Hub...")
    start = time.time()
    !docker pull {FULL_IMAGE}
    elapsed = time.time() - start
    print(f"Pull complete in {elapsed:.0f} seconds.")

## Step 4: Launch Container with GPU

Runs Jupyter inside the container with GPU passthrough (`--gpus all`).
The container mounts:
- Repo code at `/workspace`
- Google Drive at `/content/drive`

In [ ]:
CONTAINER_NAME = "lightningposetrack"

# Stop existing container if any
!docker rm -f {CONTAINER_NAME} 2>/dev/null || true

# Launch container with GPU
!docker run -d --name {CONTAINER_NAME} \
    --gpus all \
    -p {JUPYTER_PORT}:8888 \
    -v {REPO_DIR}:/workspace \
    -v /content/drive:/content/drive \
    -w /workspace \
    {FULL_IMAGE} \
    jupyter notebook --ip=0.0.0.0 --port=8888 --no-browser --allow-root \
        --NotebookApp.token='' --NotebookApp.password='' \
        --NotebookApp.allow_origin='*'

print("Container launched. Checking Jupyter startup...")
!sleep 3 && docker logs {CONTAINER_NAME} 2>&1 | tail -5

## Step 5: Access Jupyter & Verify GPU

Since Colab cannot directly open the forwarded port, use **ngrok** or **Cloudflare Tunnel** for external access.

Alternatively, run notebooks directly in this Colab session (the standard workflow).
The container is useful for:
- Running `lightning-pose[all]` in a reproducible environment
- Testing code changes before pushing
- Connecting VS Code via Remote - Containers

In [ ]:
print("=" * 60)
print("Container Status:")
!docker ps --filter name={CONTAINER_NAME} --format "table {{.Names}}\t{{.Status}}\t{{.Ports}}"

print("\nGPU accessible inside container:")
!docker exec {CONTAINER_NAME} python -c "import torch; print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}, Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"N/A\"}')"

print("\n--- Container ready ---")
print("\nTo connect VS Code:")
print("  1. Install 'Dev Containers' extension")
print("  2. Ctrl+Shift+P → Dev Containers: Attach to Running Container")
print("  3. Select 'lightningposetrack' container")

## Appendix: Useful Commands

```bash
# Exec into the container
docker exec -it lightningposetrack bash

# View Jupyter logs
docker logs lightningposetrack

# Stop the container
docker stop lightningposetrack

# Remove the container
docker rm -f lightningposetrack
```